# Phase 2: Data Acquisition & Preprocessing (PRONOSTIA/FEMTO)

This notebook implements the data acquisition, feature extraction, and feature selection pipeline for the PRONOSTIA dataset using the `rul-datasets` library. 

**Pipeline Specifications:**
1.  **Global Configuration**: Setup for thresholds, window sizes, and paths.
2.  **Data Acquisition**: Load FEMTO data via `rul-datasets` and concatenate across conditions (Cross-Condition).
3.  **Feature Extraction**: Extract 124 features per window (2560 samples) using the ConstructHI methodology.
4.  **Feature Selection**: Compute Spearman Correlation and Modified Monotonicity on the Training set. Select features based on a threshold.
5.  **Normalization**: Fit Min-Max scaling on the training set and transform all sets.
6.  **Visualization & Evaluation**: Display selected features and plot their degradation against target RUL.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import scipy.stats as stats
import scipy.signal
from scipy.integrate import cumulative_trapezoid
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import warnings

# We need the rul-datasets library for Data Acquisition
try:
    import rul_datasets
except ImportError:
    print("Warning: rul_datasets is not installed. Run 'pip install rul-datasets'.")

warnings.filterwarnings('ignore')

## 1. Global Configuration
Setting up thresholds, extractors, and constants for the pipeline.

In [ ]:
# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
FEATURE_EXTRACTOR_PATH = r"D:\Proyek Dosen\Riset Bearing\Notebook-Github\3rd Research_Cross-Domain Generalization RUL Bearing with XAI\src\Nemani_ConstructHI_Method"
THRESHOLD_SELECTION = 0.4
WINDOW_SIZE = 2560

# Add Feature Extractor class path
sys.path.append(FEATURE_EXTRACTOR_PATH)
try:
    from FeatureExtractionAndSelection import FeatureExtractionAndSelection
except ImportError:
    print("Warning: Could not import FeatureExtractionAndSelection.")

## 2. Data Acquisition using `rul-datasets`
Loading train (dev), validation (val), and test splits across different operating conditions (fd=1, 2, 3), ensuring cross-condition generalization.

In [ ]:
def load_femto_data():
    """Loads all FEMTO conditions and returns lists of bearings to preserve per-bearing structure."""
    dev_X_list, dev_y_list = [], []
    val_X_list, val_y_list = [], []
    test_X_list, test_y_list = [], []
    
    # Store condition frequency info for each bearing
    dev_freqs, val_freqs, test_freqs = [], [], []
    
    for fd in [1, 2, 3]:
        print(f"Loading FEMTO dataset fd={fd}...")
        reader = rul_datasets.reader.FemtoReader(fd=fd, norm_rul=True)
        
        dev_X, dev_y = reader.load_split("dev")
        val_X, val_y = reader.load_split("val")
        test_X, test_y = reader.load_split("test")
        
        dev_X_list.extend(dev_X)
        dev_y_list.extend(dev_y)
        dev_freqs.extend([fd] * len(dev_X))
        
        val_X_list.extend(val_X)
        val_y_list.extend(val_y)
        val_freqs.extend([fd] * len(val_X))
        
        test_X_list.extend(test_X)
        test_y_list.extend(test_y)
        test_freqs.extend([fd] * len(test_X))
        
    print("\nData Loading Complete.")
    print(f"Total Train Bearings: {len(dev_X_list)}")
    print(f"Total Val Bearings: {len(val_X_list)}")
    print(f"Total Test Bearings: {len(test_X_list)}")
    
    return {
        'train': (dev_X_list, dev_y_list, dev_freqs),
        'val': (val_X_list, val_y_list, val_freqs),
        'test': (test_X_list, test_y_list, test_freqs)
    }

# Execution
if 'rul_datasets' in sys.modules:
    femto_data = load_femto_data()
else:
    print("Mocking data to prevent execution failure if dataset missing.")
    femto_data = {
        'train': ([np.random.randn(100, WINDOW_SIZE, 2)], [np.linspace(1, 0, 100)], [1]),
        'val': ([np.random.randn(50, WINDOW_SIZE, 2)], [np.linspace(1, 0, 50)], [1]),
        'test': ([np.random.randn(50, WINDOW_SIZE, 2)], [np.linspace(1, 0, 50)], [1])
    }

## 3. Feature Extraction
Extracting 124 robust statistics and frequency indicators per observation window.

In [ ]:
# Initialize Class using dataset_name
extractor = FeatureExtractionAndSelection(data_directory="dummy", dataset_name="PRONOSTIA")

def extract_features_for_split(X_list, fd_list, extractor):
    """Iterates through each bearing and extracts features using proper condition shaft frequencies."""
    extracted_features_list = []
    
    for i, X_raw in enumerate(X_list):
        fd = fd_list[i]
        shaft_freq = extractor.shaft_frequencies[fd]
        
        # Directly call object method (No logic reimplemented in notebook!)
        features = extractor.extract_features_from_array(X_raw, shaft_freq=shaft_freq)
        extracted_features_list.append(features)
        
    return extracted_features_list

print("Extracting features for Training Set...")
X_train_ext_list = extract_features_for_split(femto_data['train'][0], femto_data['train'][2], extractor)
y_train_list = femto_data['train'][1]

print("Extracting features for Validation Set...")
X_val_ext_list = extract_features_for_split(femto_data['val'][0], femto_data['val'][2], extractor)
y_val_list = femto_data['val'][1]

print("Extracting features for Test Set...")
X_test_ext_list = extract_features_for_split(femto_data['test'][0], femto_data['test'][2], extractor)
y_test_list = femto_data['test'][1]

# Optional: verify shape
print(f"Extracted Train Bearing 1 Shape: {X_train_ext_list[0].shape}")
}

## 4. Feature Selection (Spearman & Monotonicity on Train Set)
Selecting features that highly correlate chronologically and monotonically with linear degradation target.

In [ ]:
print("Running criteria evaluation on Train set per bearing...")

# Using the class method to compute criteria (no inline rewriting of Spearman/Monotonicity formulas)
metrics_df = extractor.calculate_criteria(X_train_ext_list, y_train_list)

# Filtering with threshold
selected_features_df = metrics_df[metrics_df['Criteria'] > THRESHOLD_SELECTION]
selected_indices = selected_features_df['Feature_Idx'].values

print(f"Total Selected Features (> {THRESHOLD_SELECTION}): {len(selected_indices)} / 124")

# Helper to concatenate lists of arrays and drop unselected features
def process_and_concat(X_list, y_list, sel_idx):
    X_concat = np.concatenate(X_list, axis=0)[:, sel_idx]
    y_concat = np.concatenate(y_list, axis=0)
    return X_concat, y_concat

# Concatenating & Dropping Unselected Features
X_train_sel, y_train_concat = process_and_concat(X_train_ext_list, y_train_list, selected_indices)
X_val_sel, y_val_concat = process_and_concat(X_val_ext_list, y_val_list, selected_indices)
X_test_sel, y_test_concat = process_and_concat(X_test_ext_list, y_test_list, selected_indices)
}

## 5. Normalization (Scaling)
To prevent scale biasing, scaling the features locally bounded by standard extremes, computed exclusively via the Train Set parameters.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))

# Fit directly on selected train
scaler.fit(X_train_sel)

# Transform the branches
X_train_norm = scaler.transform(X_train_sel)
X_val_norm = scaler.transform(X_val_sel)
X_test_norm = scaler.transform(X_test_sel)

print(f"Normalized Train Shape: {X_train_norm.shape}")

## 6. Visualization & Evaluation
Showcase selected indices mathematically and visually demonstrate the progression alignment for random sample validations.

In [ ]:
print(f"Total Selected Features: {len(selected_indices)}")
print(f"Selected List:\n{list(selected_indices)}\n")

plt.figure(figsize=(15, 5))
colors = ['green' if cri > THRESHOLD_SELECTION else 'gray' for cri in metrics_df['Criteria']]
plt.bar(metrics_df['Feature_Idx'], metrics_df['Criteria'], color=colors)
plt.axhline(y=THRESHOLD_SELECTION, color='red', linestyle='--', linewidth=2, label=f'Threshold = {THRESHOLD_SELECTION}')
plt.title("Feature Criteria (Spearman + Monotonicity avg) Evaluated on Train Set")
plt.xlabel("Feature Index")
plt.ylabel("Criteria Score")
plt.legend()
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Linear RUL vs Top Feature (from Val set)
if len(selected_indices) > 0:
    # Use iloc[0]['Feature_Idx'] to get the actual feature index, avoiding pandas index conflicts
    top_feature_global_idx = int(
        selected_features_df.sort_values(by='Criteria', ascending=False).iloc[0]['Feature_Idx']
    )
    
    # Local index inside the normalized matrix
    top_feature_pos = list(selected_indices).index(top_feature_global_idx)
    
    # We plot the first bearing's test data natively (no slicing needed, it's just the first bearing from Val Set list)
    plot_limit = len(y_val_list[0]) 
    
    val_target_sample = y_val_list[0]
    val_feature_sample = X_val_norm[:plot_limit, top_feature_pos]
    
    fig, ax1 = plt.subplots(figsize=(10, 4))

    color = 'tab:blue'
    ax1.set_xlabel('Time (Steps)')
    ax1.set_ylabel('Linear RUL Target', color=color)
    ax1.plot(val_target_sample, color=color, linestyle='--', linewidth=2, label='Target Linear RUL')
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()  
    color = 'tab:orange'
    ax2.set_ylabel(f'Top Feature (Global Idx: {top_feature_global_idx})', color=color)  
    ax2.plot(val_feature_sample, color=color, linewidth=2, alpha=0.8, label='Selected Feature')
    ax2.tick_params(axis='y', labelcolor=color)

    fig.tight_layout()  
    plt.title("Linear Degradation Ground Truth vs Best Evaluated Feature (First Validation Bearing)")
    ax1.grid(True)
    plt.show()
else:
    print("No features met the threshold.")